In [ ]:
from email.message import EmailMessage
import os
import smtplib
import requests
from openai.types.responses import ResponseTextDeltaEvent
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, SQLiteSession, OpenAIChatCompletionsModel, set_tracing_disabled
from IPython.display import Markdown, display

set_tracing_disabled(disabled=True)

ollama_client = AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
local_llm = OpenAIChatCompletionsModel(model="llama3.2", openai_client=ollama_client)

smtp_server = os.getenv("EMAIL_SMTP_SERVER")
mail_app_password = os.getenv("EMAIL_APP_PASSWORD")
email_address = os.getenv("EMAIL_ADDRESS")
USE_EMAIL = True

def send_email(subject, text_body, html_body):
    msg = EmailMessage()
    msg["From"] = email_address
    msg["To"] = email_address
    msg["Subject"] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype="html")

    with smtplib.SMTP(smtp_server, 587) as server:
        server.starttls()
        server.login(email_address, mail_app_password)
        server.send_message(msg)

@function_tool
def send_message(subject, text_body, html_body):
    """ Send the message either to the email or print based on the USE_EMAIL variable """
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        print(f"Subject: {subject}\n\n{text_body}")

